# Week 12C: A full-stack county Census tract explorer

Choose a **state and county**, prepare its Census data, and visualize population, income, poverty, and home value **at the census tract level**. Every colored polygon, table row, and exported record represents one tract. County names select the study area; they are not the mapped statistical unit.

[Download this notebook](lecture-12C.ipynb) · [Runnable app](downloads/census-full-stack.zip) · [Tract-only dataset ZIP](downloads/census-tract-data.zip) · [App README](examples/census-full-stack/README.md) · [Previous: publishing](lecture-12B.html)

## Quick start

Open a terminal in the folder containing this notebook:

```bash
conda activate geospatial
conda install -c conda-forge fastapi uvicorn httpx shapely
cd examples/census-full-stack
python -m uvicorn app:app --host 127.0.0.1 --port 8000
```

For the standalone ZIP, use `cd census-full-stack`. Keep the terminal running and open **[http://127.0.0.1:8000/](http://127.0.0.1:8000/)**.

1. Select **Pennsylvania**. The **County** dropdown loads automatically.
2. Select **Dauphin County (`42043`)**, then click **Prepare census tract data**.
3. Select a Census variable. Click a tract or table row to inspect its estimates and uncertainty.
4. Select **New Jersey → Mercer County (`34021`)** and prepare its tracts.
5. Use **Prepared county** to switch between the saved county tract datasets.

The checked 2024 data contain **67 tracts in Dauphin County** and **84 in Mercer County**. The generated dataset includes both counties; a fresh app installation downloads a county when you prepare it. Stop the old server and hard-refresh after upgrading from earlier versions.

**Expected result:** OpenStreetMap streets, tract choropleth fills, clear tract boundaries, a legend, tract details, and a tract-level table. The selected tract is outlined in orange. The map does not display county-level aggregate values.

## 1. Understand the full stack

| Layer | Technology | Responsibility |
| --- | --- | --- |
| Frontend | Leaflet, HTML, CSS, JavaScript | State/county selection, variable selector, tract choropleth, popup, table, downloads |
| Backend | Python, FastAPI, Uvicorn, HTTPX | Fetch public data, select all tracts by county FIPS, join ACS estimates by tract GEOID |
| Database | SQLite, bundled with Python | Save prepared counties and individual tract records between restarts |
| Public data | Census TIGERweb and Census Reporter | Matching 2024 geography and 2020–2024 ACS five-year estimates |

```text
Browser: select state → GET /api/counties?state=42
Browser: select county → POST /api/prepare {"county_id":"42043"}
Python: tract boundaries for state 42 + county 043 → ACS tables → join by 11-digit tract GEOID
SQLite: save one record per tract under its parent county
Browser: GET /api/tracts?county_id=42043&variable=income → tract choropleth and table
```

The frontend and API share http://127.0.0.1:8000, so no separate frontend server or CORS setup is needed. Part A uses DuckDB Spatial for synthetic teaching dots; this app stores real ACS tract estimates in SQLite. Shapely repairs/simplifies display geometries. County membership comes from Census geographic codes, not a spatial overlap with a municipality.

## 2. Project and data folders

```text
lecture folder/
├── lecture-12C.ipynb
├── data/
│   ├── census-tracts/
│   │   ├── acs2024_tracts.geojson       # one feature per unique tract
│   │   ├── acs2024_tracts.csv           # same attributes, no geometry
│   │   ├── counties/42043_tracts.geojson
│   │   └── README.txt                  # data dictionary and limitations
│   ├── county-lists/42.json            # county names for the selected state
│   └── tract-explorer/                 # raw response cache and preparation packages
└── examples/census-full-stack/
    ├── app.py
    ├── database.py
    ├── datasets.py
    ├── prepare_data.py
    ├── test_app.py
    ├── static/
    └── data/tracts.sqlite              # persistent database
```

Use the existing `data` folder beside the notebook. The old synthetic point and district files remain inputs to Part A only. The deliverable tract dataset contains no town polygons or county aggregates.

For the standalone app ZIP, extract `census-full-stack` beside the notebook. Data downloads go into the same sibling `data` folder automatically. Extract the tract-data ZIP into the lecture folder to obtain the ready-made `data/census-tracts` files.

In [ ]:
from pathlib import Path

lecture_dir = Path.cwd()
app_dir = lecture_dir / "examples" / "census-full-stack"
if not app_dir.is_dir():
    app_dir = lecture_dir / "census-full-stack"
assert (app_dir / "app.py").is_file(), "Open this notebook from its lecture folder."
print("Application:", app_dir.resolve())
print("Tract datasets:", (lecture_dir / "data" / "census-tracts").resolve())

## 3. Install and start the app

Keep using the course **`geospatial`** environment. Install `fastapi uvicorn httpx shapely` using the quick-start command above. SQLite is included with Python; no database server or Spatial extension installation is needed. Select your `geospatial` notebook kernel and restart it after installing packages.

Start Uvicorn from the app directory in **Terminal 1**:

```bash
python -m uvicorn app:app --host 127.0.0.1 --port 8000
```

Open the [app](http://127.0.0.1:8000/), [API documentation](http://127.0.0.1:8000/docs), and [health check](http://127.0.0.1:8000/api/health).

In **Terminal 2**, verify:

```bash
curl http://127.0.0.1:8000/api/health
```

Expect `status: "ok"`, `year: 2024`, `geography_level: "census_tract"`, and counts for `prepared_counties` and `prepared_tracts`. Fresh installations begin at zero. The database is created automatically; preparation happens through the browser.

Keep the server running in its terminal, not in a notebook cell. Opening `index.html` directly or using a static server does not run the API. Stop with **Ctrl+C**. After upgrading, restart Uvicorn and hard-refresh (Mac: Cmd+Shift+R; Windows/Linux: Ctrl+Shift+R).

## 4. Choose counties in different states

### Pennsylvania: Dauphin County

Select **Pennsylvania**, then **Dauphin County (`42043`)**. Click **Prepare census tract data**. The backend downloads all 67 assigned tracts and their ACS estimates. Choose a variable to map differences between those tracts.

### New Jersey: Mercer County

Change the state to **New Jersey**. Its county list replaces the Pennsylvania list automatically. Select **Mercer County (`34021`)** and prepare its 84 tracts. Both datasets now appear under **Prepared county**, labeled with their state names.

### Other counties

Choose any available state and county. No typed name is required. The list contains Census **counties and county equivalents**: some states use parishes, boroughs, census areas, or independent cities at this geography level. These are county-level selections, not towns, townships, places, or county subdivisions.

The five-digit county identifier combines a two-digit state FIPS and three-digit county FIPS. Tract GEOIDs begin with that same five-digit county identifier. Example: `42043` identifies the parent county; an 11-digit code such as `42043020100` identifies an individual tract within it.

Tracts nest in counties. Preparation requests **all tracts with the matching state and county codes**; it no longer selects tracts by municipality overlap. Repreparing replaces one county atomically, reuses fixed-vintage cached responses, and retains other counties. Previous town tables remain separate from the active county tables and never appear in the county selector.

## 5. Read the tract choropleth

| Variable | ACS table | Meaning |
| --- | --- | --- |
| Total population | B01003 | Estimated residents of the tract; count, not density |
| Median household income | B19013 | Median in 2024 inflation-adjusted dollars |
| Poverty rate | B17001 | Below-poverty count / poverty-status universe × 100 |
| Median owner-occupied home value | B25077 | Median owner-occupied unit value, in 2024 dollars |

Every polygon is a census tract. Class breaks stay fixed per variable when switching counties, so a color has the same value range in each county. The orange outline identifies the selected tract. Click a tract or its GEOID in the table to inspect all measures. Gray means unavailable; true zeros remain zero.

Direct estimates show available **90% margins of error (MOEs)**. Poverty displays numerator and denominator; a rate MOE is not calculated. `+` flags upper-coded medians (income $250,000+ and home value $2,000,000+). Large MOEs indicate uncertainty.

These are **2020–2024 ACS five-year survey estimates**, not current counts or synthetic dots. Medians cannot be added across tracts. County selection controls which tracts appear; the app does not replace tract values with a county total. Geometry is simplified slightly for display after selection by geographic code.

## 6. Download and inspect tract-only data

Use **Download tract GeoJSON** or **Download tract CSV** to obtain the selected county's tract records.

- [Combined GeoJSON](data/census-tracts/acs2024_tracts.geojson): one polygon feature per tract across prepared counties.
- [Combined CSV](data/census-tracts/acs2024_tracts.csv): the same attributes without geometry.
- [Dataset ZIP](downloads/census-tract-data.zip): the ready-made county tract files and data dictionary.
- `data/census-tracts/counties/<county-fips>_tracts.geojson` and `.csv`: county-specific tract subsets.

Fields include `geoid`, `tract_name`, `state_fips`, `county_fips`, `tract_code`, `county_name`, `county_geoid`, `acs_year`, `population`, `income`, `poverty`, `home_value`, uncertainty values, and top-coding flags. `county_geoid` is the five-digit parent county code; `geoid` is the 11-digit tract code. Preserve GEOID/FIPS as text when opening the CSV.

The prepared download covers Dauphin County, PA, and Mercer County, NJ; it is not a nationwide tract dataset. Prepare more counties to expand the local export. Every row remains a tract; county names are parent-geography labels, not aggregate observations. Missing estimates are null in GeoJSON and blank in CSV.

Public response caches and preparation internals stay in `data/tract-explorer`, separate from the deliverable dataset. County-name catalogs are in `data/county-lists`.

## 7. Optional command-line preparation

From the app directory:

```bash
conda activate geospatial
python prepare_data.py --state 42 --list-counties
python prepare_data.py --county 42043
python prepare_data.py --county 34021
python prepare_data.py --export
```

`--list-counties` returns the county names and FIPS codes for a state. `--county` prepares the complete tract set for that county. `--export` writes tract-only files from the current database without new downloads. Refresh the app after CLI preparation.

`CENSUS_DB` overrides the SQLite path; `CENSUS_DATA_DIR` overrides the internal cache/package folder. Use the same settings in both terminals. The deliverable dataset and county-list folders are siblings of the cache folder. Never point SQLite at an old DuckDB file.

## 8. Inspect the API contract

| Request | Response |
| --- | --- |
| `GET /api/health` | Status, vintage, prepared county and tract counts |
| `GET /api/config` | Variables, units, source descriptions |
| `GET /api/counties?state=42` | County names and five-digit county IDs for Pennsylvania |
| `POST /api/prepare` with `{"county_id":"42043"}` | Prepare complete tract dataset for one county |
| `GET /api/prepared-counties` | Saved counties, states, and tract counts |
| `GET /api/tracts?county_id=42043&variable=income` | Tract GeoJSON for the choropleth |
| `GET /api/dataset?county_id=42043&format=geojson` | Flat tract-only download; use `format=csv` for CSV |

Omit `county_id` from `/api/dataset` to export all unique prepared county tracts. Supported variable keys are `population`, `income`, `poverty`, and `home_value`.

In browser developer tools → **Network**, inspect the county-list GET, preparation POST, and tract GET. In `/docs`, prepare a county and then try `/api/tracts`.

The following optional client requires Uvicorn to be running and Dauphin County prepared first.

In [ ]:
import httpx

response = httpx.get(
    "http://127.0.0.1:8000/api/tracts",
    params={"county_id": "42043", "variable": "income"},
    timeout=30,
)
response.raise_for_status()
tracts = response.json()
print(tracts["county_name"], "—", len(tracts["features"]), "census tracts")
[
    {"geoid": f["properties"]["geoid"], "income": f["properties"]["value"]}
    for f in tracts["features"][:5]
]

## 9. Trace Python and SQL

[prepare_data.py](examples/census-full-stack/prepare_data.py) uses the **Census TIGERweb ACS 2024 service**. Layer 82 lists counties; layer 8 supplies tracts. Both queries are paginated. A county request filters by `STATE` and `COUNTY`; every returned tract must start with that county's five-digit FIPS. No town boundary or spatial intersection is used.

The script joins tract geometries to four ACS tables from Census Reporter's `acs2024_5yr` endpoint by full tract GEOID. It checks the returned release and requires every selected tract to be present before replacing saved data. Invalid Census sentinel values become null, not zero; a zero poverty denominator yields no rate.

[database.py](examples/census-full-stack/database.py) stores parent-county metadata in `counties` and individual tract records in `county_tracts`. A transaction replaces one county while preserving others. Legacy town tables stay untouched and are excluded from county reads.

[app.py](examples/census-full-stack/app.py) validates parameters and uses bound SQL values:

```sql
SELECT * FROM county_tracts WHERE county_id = ? ORDER BY geoid
```

[datasets.py](examples/census-full-stack/datasets.py) writes flat tract attributes and unique GEOIDs. It includes parent-county labels but no county polygons or county-level aggregate records. Source responses, release metadata, and county selection information are retained for reproducibility.

## 10. Trace the frontend and test the app

[static/app.js](examples/census-full-stack/static/app.js) loads county names whenever the state changes. Preparing a county sends JSON to FastAPI. The selected variable controls tract fills and the legend; each tract also has a popup and table row. Selecting a row zooms to that tract and highlights its boundary. Request counters discard outdated map/list responses.

Run from the app directory:

```bash
python -m unittest -v
```

Eight offline tests use synthetic fixtures and temporary databases. They verify complete county selection by FIPS, rejection of a wrong-county tract, state county lists, release mismatch, incomplete-download protection, preservation of other counties and legacy town data, missing values, uncertainty, and tract-only exports.

Then test the browser:

1. Select Pennsylvania; confirm only county-level names appear, not towns or townships.
2. Choose Dauphin County and prepare it; verify 67 tract polygons and table rows.
3. Switch all four variables and inspect a tract popup.
4. Change to New Jersey; verify its county list replaces Pennsylvania's.
5. Prepare Mercer County; switch between saved counties and verify the viewport and labels change.
6. Download CSV/GeoJSON; check each tract GEOID begins with the selected county code.
7. Restart Uvicorn; confirm saved county datasets remain available.

## 11. Troubleshooting and hosting

| Symptom | Check |
| --- | --- |
| Missing FastAPI, HTTPX, or Shapely | Activate `geospatial` and install the quick-start packages |
| Town selector still appears | Stop the old Uvicorn process, start the updated app, and hard-refresh |
| County list unavailable | Verify public-service connectivity and use **Reload county list** |
| No prepared county | Select state/county and click **Prepare census tract data** |
| Preparation fails | Check Census TIGERweb/Census Reporter connectivity and the server terminal; previous saved county data remain available |
| Blank map library | Allow `unpkg.com` through the network/content blocker |
| Tracts visible but streets blank | Check `tile.openstreetmap.org` requests and internet access |
| Gray tract | Estimate unavailable or suppressed; inspect its details rather than assuming zero |
| Port occupied | Stop the previous server or choose `--port 8001` and use that URL everywhere |
| File is not a database | Unset a `CENSUS_DB` override pointing at an old DuckDB file |

The app needs no API key: Census Reporter's public ACS mirror supplies estimates, Census TIGERweb supplies geography, and free OpenStreetMap supplies tiles. First-time downloads, Leaflet, and map tiles require internet access. Prepared estimates persist locally.

Stop the server before manually removing database/cache files. To refresh fixed-vintage source responses, remove only `data/tract-explorer/source-cache`, restart, and reprepare a county. Prior municipality exports are archived under `data/tract-explorer/legacy-town-exports`; they are not included in the county data download.

Quarto and GitHub Pages publish the lecture but cannot run FastAPI. A deployed app needs a Python-capable host, persistent SQLite/download storage, HTTPS, and separate access/resource controls. Use one server worker for this classroom demo. Running locally does not publish anything.

Render the notes from the lecture folder:

```bash
quarto render lecture-12C.ipynb --to html --no-execute
```

## Exercises and sources

1. Compare tract incomes in two counties using the same legend. Explain why color does not represent density.
2. Split a tract GEOID into state, county, and tract codes. Verify its parent county.
3. Recalculate one tract's poverty rate from the numerator and denominator.
4. Inspect a direct estimate and its MOE. Discuss how uncertainty affects interpretation.
5. Prepare a county in another state and download its tract-only dataset.

Sources: [Census TIGERweb ACS 2024 layers](https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_ACS2024/MapServer), [2024 ACS five-year geography](https://api.census.gov/data/2024/acs/acs5/geography.html), [Census Reporter API](https://github.com/censusreporter/census-api/blob/master/API.md), [OpenStreetMap attribution](https://www.openstreetmap.org/copyright).

Estimates are real ACS survey estimates. The older synthetic historical point/district data remain part of Part A, not this county tract explorer.